### 🔰PyTorchでニューラルネットワーク基礎 #35 【可変長テキスト分類】

### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください


### データについて
* history_text_label_id.jsonl: wikipediaの日本の歴史分野の内容から収集した時代区分を判定するテキスト
    * ids列の長さがそれぞれことなります。
    * 時代区分を直接表すテキストは除いてある。（江戸時代ラベルの文章に「江戸」という単語がはありません）
    * クラス数：５（弥生、奈良、室町、江戸、昭和）

### TransformerEncoderを利用したBERTタイプの文章分類のネットワークの構造について

* 事前学習はseq_len=64で学習済み
* 最大の系列長が64となる

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer

# カスタマイズする部分
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split

In [2]:
#デバイスの選択
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("利用デバイス:", device.type)

# 精度を計算する関数
def accuracy(y, t):
    _, argmax_list = torch.max(y, dim=1)
    accuracy = sum(argmax_list == t).item()/len(t)
    return accuracy

利用デバイス: cuda


In [3]:
pre_trained_model = "./model/unigram_2k.model"                # 事前学習されたモデル
data_filename = "./data/history_text_label_id.jsonl"          # 分類問題のデータ
tokenizer_filename = "tokenizer/unigram_tokenizer_2k.json"    # 保存したトークナイザーで確認

df = pd.read_json(data_filename, lines=True)
tokenizer = Tokenizer.from_file(tokenizer_filename)

## データの系列長がそれぞれ異なる
* バッチ学習時に\<pad\>して揃えるcollate_fn関数やDatasetクラスをカスタマイズしていく

In [ ]:
df.sample(3)

,text,label,ids
501,同月には江川英龍や高島秋帆に西洋流砲術を導入させ、近代軍備を整えさせた。,2,"[1, 301, 250, 50, 473, 954, 3, 16, 95, 804, 3,..."
838,これをきっかけに将軍の力は衰えた。,4,"[1, 152, 7, 712, 9, 47, 5, 91, 11, 1928, 48, 2..."
458,このような風潮は「山師、運上」という言葉で語られ、利益追求型で場当たり的な面が多く、腐敗も目...,2,"[1, 51, 354, 31, 674, 3, 11, 67, 72, 179, 8, 4..."


idsの要素数がそれぞれ異なることを確認してみた

In [5]:
[len(df.iloc[i]["ids"]) for i in range(10)]

[12, 22, 13, 64, 14, 55, 26, 38, 34, 64]

In [6]:
# 訓練データと検証データに分割
train_data, test_data = train_test_split(df, stratify=df["label"], random_state=55)

**train_dataのidsとlabelに対して行う処理**
* カスタムDatasetクラス：train_dataのidsとlabelを辞書形式で出力させたい
* DataLoaderクラス：カスタムDatasetで生成されたデータをバッチサイズごとに\<pad\>で等長化

**必要な作業**
* Datasetクラスの定義
* collate_fnの定義

In [7]:
class SimpleDataset(Dataset):
    def __init__(self, data):
        self.ids = data["ids"].tolist()
        self.label = data["label"].tolist()
   
    def __len__(self):
        return len(self.label)
    
    # 入力するデータのタイプによって適宜修正
    def __getitem__(self, idx):
        return {"ids":   self.ids[idx],
                "label": self.label[idx]}

**collate_fnの定義**
* バッチ内の最長の系列長に合わせて、各idsに\<pad\>IDの 0 を挿入
* collate関数の引数batchは``dataset.__getitem__(idx)``のリスト
```python
[{'ids':tensor(), 'label':tensor()}, {'ids':tensor(), 'label':tensor()},... ]
```


In [8]:
def padding_collate_fn(batch):
    # (1) batchはデータフレーム、tensor化する
    ids_list = [torch.tensor(item["ids"], dtype=torch.long) for item in batch]
    labels = [item["label"] for item in batch]
    
    # (2) <pad>の挿入
    padded_ids = pad_sequence(ids_list, batch_first=True, padding_value=0)
    attention_masks = (padded_ids != 0).long()   # 今回使わないけどpad部分をマスクするattention_maskも作れるぞ〜
  
    # (3) DataLoaderの出力
    return {
        "ids": padded_ids,
        "attention_mask": attention_masks,
        "label": torch.tensor(labels, dtype=torch.long)
    }

In [ ]:
train_dataset = SimpleDataset(train_data)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    collate_fn = padding_collate_fn,
    shuffle=True,
    drop_last=True
)

In [10]:
# 実際に学習されるバッチごとに系列長が異なることを確認
print([x["ids"].shape[1] for x in train_loader])
#for x in train_loader:
#    print(x["ids"])

[64, 64, 64, 64, 60, 64, 64, 64, 64, 64, 64, 64, 64, 64, 60, 64, 64, 55, 62, 64, 64, 58, 64, 64, 64, 64, 64, 64, 64, 64, 64, 61, 64, 64, 64, 64, 50, 60, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 53, 64, 64, 64, 64, 64, 64, 64, 41, 64, 59, 52, 64, 64, 64, 64, 64, 64, 58, 64, 64, 64, 64, 64, 64, 60, 64, 64, 64, 64, 64, 64, 64, 56, 64, 58, 64, 64, 58, 64]


**モデルの設定**
* 事前学習時に使っていたmlm_head部分を分類問題用Linearに修正
* 文頭\<bos\>の特徴量を分類問題用Linearへ入力できるように修正

In [11]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.d_model = 64
        self.seq_len = 64
        self.nhead = 4
        self.dim_feedforward = 256
        self.num_layers = 6
        self.dropout = 0.1
        self.out_features = 5
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.mask_token_id = tokenizer.token_to_id("<mask>")
        self.bos_token_id = tokenizer.token_to_id("<bos>")
        self.eos_token_id = tokenizer.token_to_id("<eos>")
        self.unk_token_id = tokenizer.token_to_id("<unk>")

        # 特殊トークンのセット
        self.special_tokens = {
            self.pad_token_id,
            self.mask_token_id,
            self.bos_token_id,
            self.eos_token_id,
            self.unk_token_id,
        }
        
        # 通常トークンのリスト special_tokenを除くトークンのリスト（MLMランダム置換用）
        self.normal_tokens = [
            i for i in range(self.vocab_size) 
            if i not in self.special_tokens
        ]
        
        # 学習設定 (今回は利用しないけど使うと便利かも)
        self.batch_size = 512
        self.learning_rate = 0.0001
        self.num_epochs = 100
        self.mask_prob = 0.15
        self.max_grad_norm = 1.0


class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        # 埋め込み層
        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size, 
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id
        )
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # MLM用の出力層：BERT風レイヤー　各トークン位置で語彙全体を予測
        self.classifier = nn.Linear(in_features=config.d_model, out_features=config.out_features)

    
    def forward(self, x):
        # マスクの作成
        src_key_padding_mask = (x == self.config.pad_token_id)
        
        # 埋め込み
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(torch.arange(x.size(1), device=x.device))
        x = tok_emb + pos_emb.unsqueeze(0)
        
        x = self.layer_norm(x)
        x = self.dropout(x)
        
        # Transformer Encoder
        h = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)

        # 文ベクトルへの Pooling  <BOS>トークン（先頭）に情報を集約
        pooled = h[:, 0, :]  # [batch, d_model]
        
        # 分類 
        y = self.classifier(pooled)  # [batch, num_labels=5]    
        return y

## 学習済みモデルの読み込み

In [12]:
# 事前学習モデルの設定を分類問題用へ更新
checkpoint = torch.load(pre_trained_model, map_location=device)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])

model = DNN(config).to(device)

In [13]:
# 事前学習した重みを分類問題用へコピー
# 事前学習重み（checkpoint側）をフィルタ
pretrained = checkpoint["model_state_dict"]
pretrained_filtered = {
    k: v for k, v in pretrained.items()
    if (not k.startswith("mlm_head."))      # and (not k.startswith("classifier."))
}

# 実際にコピーする部分
# strict=Falseにすることで、ファイルにはないパラメータは読み込まれず、初期状態のまま保持される
# Missing Keys: 重み付けされないパラメータ
info = model.load_state_dict(pretrained_filtered, strict=False)

# 空のリストが表示されるならコピー成功！
print(f"削除されているか確認: {info.unexpected_keys}")

削除されているか確認: []


In [14]:
params_to_update = []

for name , param in model.named_parameters():
    param.requires_grad = False
for name, param in model.transformer_encoder.layers[-2:].named_parameters():  # 最終層＋１
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name)
for name, param in model.classifier.named_parameters():
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name) 

更新されるパラメータ: 0.self_attn.in_proj_weight
更新されるパラメータ: 0.self_attn.in_proj_bias
更新されるパラメータ: 0.self_attn.out_proj.weight
更新されるパラメータ: 0.self_attn.out_proj.bias
更新されるパラメータ: 0.linear1.weight
更新されるパラメータ: 0.linear1.bias
更新されるパラメータ: 0.linear2.weight
更新されるパラメータ: 0.linear2.bias
更新されるパラメータ: 0.norm1.weight
更新されるパラメータ: 0.norm1.bias
更新されるパラメータ: 0.norm2.weight
更新されるパラメータ: 0.norm2.bias
更新されるパラメータ: 1.self_attn.in_proj_weight
更新されるパラメータ: 1.self_attn.in_proj_bias
更新されるパラメータ: 1.self_attn.out_proj.weight
更新されるパラメータ: 1.self_attn.out_proj.bias
更新されるパラメータ: 1.linear1.weight
更新されるパラメータ: 1.linear1.bias
更新されるパラメータ: 1.linear2.weight
更新されるパラメータ: 1.linear2.bias
更新されるパラメータ: 1.norm1.weight
更新されるパラメータ: 1.norm1.bias
更新されるパラメータ: 1.norm2.weight
更新されるパラメータ: 1.norm2.bias
更新されるパラメータ: weight
更新されるパラメータ: bias


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params_to_update, lr=1e-3)

In [16]:
LOOP = 40
model.train()
for epoch in range(LOOP):
    total_loss = 0
    total_acc = 0
    num_batches = 0
    
    for data in train_loader:
        optimizer.zero_grad()

        # データを取り出してデバイスに送る
        x = data["ids"].to(device)    # (batch_size, seq_length) seq_lengthは学習中のバッチごと異なる
        t = data["label"].to(device)  # (batc_size)
    
        y = model(x)
  
        loss = criterion(y,t)
        acc = accuracy(y, t)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += acc
        num_batches += 1

    if (epoch+1)%10 == 0:
        print(f"{epoch+1}:\tloss:{total_loss/num_batches:.3f}\tacc:{total_acc/num_batches:.3f}")

10:	loss:0.266	acc:0.919
20:	loss:0.184	acc:0.952
30:	loss:0.101	acc:0.974
40:	loss:0.057	acc:0.985


## 検証
* test_dataにたいしてもSimpleDatasetとDataLoaderを利用する
* 面倒なので全部を1つのバッチとして扱ってみた😆

In [17]:
# (1) テストデータ全件を 1バッチ で処理する DataLoader を作成
test_dataset = SimpleDataset(test_data)
test_all_loader = DataLoader(
    dataset = test_dataset, 
    batch_size=len(test_dataset),  # 全件を一つのバッチにする
    shuffle=False, 
    collate_fn=padding_collate_fn
)
# (2) 1回だけループを回してデータを取り出す
# next(iter(...)): よく忘れるのでメモ 
test_batch = next(iter(test_all_loader))

# (3) 検証データで推論
model.eval()
with torch.inference_mode():
    x_test = test_batch["ids"].to(device)
    t_test = test_batch["label"].to(device)

    y_test = model(x_test)
    acc = accuracy(y_test, t_test)

print(f"検証精度: {acc}")

検証精度: 0.832
